In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import re
#这个代码绝对可用！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
#这个代码绝对可用！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
#这个代码绝对可用！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
#这个代码绝对可用！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
#这个代码绝对可用！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
#这个代码绝对可用！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
#这个代码后面有个小尾巴感觉和Vertical with clim background一致，看看是不是重复了
sns.set_style("ticks")

# Suppress warnings if needed
import warnings
warnings.filterwarnings('ignore')

###########################
# Original Functions
###########################

def process_U_data(file_path, lat=60, plev=5000):
    """
    Process U data from the given file path.
    
    Parameters:
        file_path (str): Path with wildcard to the netCDF files.
        lat (int/float): Latitude to select.
        plev (int): Pressure level (in Pa) to select.
    
    Returns:
        xarray.DataArray: Processed U data.
    """
    ds = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    U = ds['U']
    U = U.mean(dim='lon')
    U = U.sel(plev=plev)
    var = U.sel(lat=lat, method='nearest')
    return var

def process_T_data(file_path, plev=5000, lat_range=(60,90)):
    """
    Process T data from the given file path.
    
    Parameters:
        file_path (str): Path with wildcard to the netCDF files.
        plev (int): Pressure level (in Pa) to select.
        lat_range (tuple): Latitude range (min_lat, max_lat) to consider.
    
    Returns:
        xarray.DataArray: Processed T data (minimum over the selected latitude range).
    """
    ds = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    T = ds['T']
    T = T.mean(dim='lon')
    T = T.sel(plev=plev)
    var = T.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    return var

def process_U_base(file_path, lat=60, plev=5000, months=[1,2,3,4,5,6], base_year='0008'):
    """
    Process the base U data.
    
    Parameters:
        file_path (str): Path to the netCDF file.
        lat (int/float): Specified latitude.
        plev (int): Pressure level (Pa).
        months (list): List of months to include.
        base_year (str): Year to select for the reference curve.
        
    Returns:
        ref (xarray.DataArray): Reference U data for the specified base_year.
        clim (xarray.DataArray): Climatology of U data.
    """
    ds = xr.open_dataset(file_path)
    U = ds['U']
    U = U.sel(plev=plev)
    U = U.sel(time=ds.time.dt.month.isin(months))
    U = U.mean(dim='lon')
    var = U.sel(lat=lat, method='nearest')
    
    # Use the specified base_year instead of hard-coded 8
    ref = var.sel(time=var.time.dt.year == int(base_year))
    clim = var.groupby("time.dayofyear").mean()
    return ref, clim

def process_T_base(file_path, plev=5000, lat_range=(60,90), months=[1,2,3,4,5,6], base_year='0008'):
    """
    Process the base T data.
    
    Parameters:
        file_path (str): Path to the netCDF file.
        plev (int): Pressure level (Pa).
        lat_range (tuple): Latitude range (min_lat, max_lat).
        months (list): List of months to include.
        base_year (str): Year to select for the reference curve.
        
    Returns:
        ref (xarray.DataArray): Reference T data for the specified base_year.
        clim (xarray.DataArray): Climatology of T data.
    """
    ds = xr.open_dataset(file_path)
    T = ds['T']
    T = T.sel(plev=plev)
    T = T.sel(time=ds.time.dt.month.isin(months))
    T = T.mean(dim='lon')
    var = T.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    
    # Use the specified base_year instead of hard-coded 8
    ref = var.sel(time=var.time.dt.year == int(base_year))
    clim = var.groupby("time.dayofyear").mean()
    return ref, clim

def plot_experiments_UT(base_ref_U, base_clim_U, base_ref_T, base_clim_T, experiments, base_year,
                        save_path, lat=60, plev_U=5000, lat_range=(60,90), plev_T=5000):
    """
    Create a figure with two panels (U and T) and plot base curves along with experiment data.
    
    Parameters:
        base_ref_U, base_clim_U: Base U reference and climatology data.
        base_ref_T, base_clim_T: Base T reference and climatology data.
        experiments (list): List of dictionaries containing experiment data and plotting parameters.
        base_year (str): Year for the base curves (e.g., "0008").
        save_path (str): File path (without extension) to save the figure.
        lat (int/float): Latitude used for U plots.
        plev_U (int): Pressure level (Pa) for U.
        lat_range (tuple): Latitude range for T plots.
        plev_T (int): Pressure level (Pa) for T.
    
    Returns:
        fig, ax: Matplotlib figure and axes array.
    """
    # Create a figure with 2 subplots (U on top, T on bottom)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 16), constrained_layout=True)
    
    # Plot base U curves
    x_base_U = range(len(base_ref_U))
    ax1.plot(x_base_U, base_ref_U, color='black', linewidth=5, label=f'Base {base_year}')
    x_clim_U = range(len(base_clim_U))
    ax1.plot(x_clim_U, base_clim_U, color='black', linestyle='--', linewidth=5, label=f'Clim {base_year}')
    
    # Plot experiment U data
    for exp in experiments:
        x_vals = range(exp['x_offset'], exp['x_offset'] + len(exp['U'].time))
        mean_U = exp['U'].mean(dim='member')
        env_min_U = exp['U'].min(dim='member')
        env_max_U = exp['U'].max(dim='member')
        ax1.plot(x_vals, mean_U, color=exp['line_color'], linewidth=3, label=exp['label'])
        ax1.fill_between(x_vals, env_min_U, env_max_U, color=exp['fill_color'], alpha=0.3)
    
    ax1.set_xlim(0, 150)
    ax1.set_xticks([0, 31, 59, 89, 120])
    ax1.set_xticklabels(['Jan','Feb','Mar','Apr','May'])
    ax1.set_ylabel(f'Zonal Mean Wind, {lat}°, {plev_U/100:.0f} hPa (m/s)', fontsize=14)
    ax1.legend(fontsize=12)
    ax1.set_ylim(-40, 80)
    
    # Plot base T curves
    x_base_T = range(len(base_ref_T))
    ax2.plot(x_base_T, base_ref_T, color='black', linewidth=5, label=f'Base {base_year}')
    x_clim_T = range(len(base_clim_T))
    ax2.plot(x_clim_T, base_clim_T, color='black', linestyle='--', linewidth=5, label=f'Clim {base_year}')
    
    for exp in experiments:
        x_vals = range(exp['x_offset'], exp['x_offset'] + len(exp['T'].time))
        mean_T = exp['T'].mean(dim='member')
        env_min_T = exp['T'].min(dim='member')
        env_max_T = exp['T'].max(dim='member')
        ax2.plot(x_vals, mean_T, color=exp['line_color'], linewidth=3, label=exp['label'])
        ax2.fill_between(x_vals, env_min_T, env_max_T, color=exp['fill_color'], alpha=0.3)
    
    ax2.axhline(y=197, color='royalblue', linestyle='--', linewidth=2)
    ax2.text(5, 194, 'Cl activation threshold', horizontalalignment='left', fontsize=12, color='royalblue')
    
    ax2.set_xlim(0, 150)
    ax2.set_xticks([0, 31, 59, 89, 120])
    ax2.set_xticklabels(['Jan','Feb','Mar','Apr','May'])
    ax2.set_ylabel(f'Minimum Temperature, {lat_range[0]}-{lat_range[1]}°N, {plev_T/100:.0f} hPa (K)', fontsize=14)
    ax2.set_xlabel('Day of Year', fontsize=14)
    ax2.legend(fontsize=12)
    ax2.set_ylim(170, 235)
    
    plt.savefig(f'{save_path}.pdf', bbox_inches='tight')
    plt.savefig(f'{save_path}.png', bbox_inches='tight')
    
    return fig, (ax1, ax2)

def assign_default_colors(experiments):
    """
    Assign default line and fill colors to experiments if not provided.
    """
    default_line_colors = ['red', 'green', 'blue', 'orange', 'purple']
    default_fill_colors = ['pink', 'lightgreen', 'lightblue', 'peachpuff', 'plum']
    
    for i, exp in enumerate(experiments):
        if 'line_color' not in exp or exp['line_color'] is None:
            exp['line_color'] = default_line_colors[i % len(default_line_colors)]
        if 'fill_color' not in exp or exp['fill_color'] is None:
            exp['fill_color'] = default_fill_colors[i % len(default_fill_colors)]
    return experiments

##############################
# Global Cache and Management
##############################

global_dataset_cache = {}

def preload_experiment_data(plots):
    """
    Pre-compute total usage count for each unique dataset key across all plots.
    """
    usage = {}
    for plot in plots:
        for exp in plot['experiments']:
            key = (exp['U_path'], exp['T_path'],
                   exp.get('lat', 60), exp.get('plev_U', 5000),
                   exp.get('lat_range', (60, 90)), exp.get('plev_T', 5000))
            usage[key] = usage.get(key, 0) + 1
    return usage

def get_experiment_data(exp, usage):
    """
    Load data for the given experiment if not already cached.
    """
    key = (exp['U_path'], exp['T_path'],
           exp.get('lat', 60), exp.get('plev_U', 5000),
           exp.get('lat_range', (60,90)), exp.get('plev_T', 5000))
    if key not in global_dataset_cache:
        u_data = process_U_data(exp['U_path'], lat=exp.get('lat', 60), plev=exp.get('plev_U', 5000)).load()
        t_data = process_T_data(exp['T_path'], plev=exp.get('plev_T', 5000), lat_range=exp.get('lat_range', (60,90))).load()
        global_dataset_cache[key] = {'u': u_data, 't': t_data, 'refcount': usage[key]}
    exp['U_data'] = global_dataset_cache[key]['u']
    exp['T_data'] = global_dataset_cache[key]['t']
    exp['data_key'] = key

def release_experiment_data(exp):
    """
    Decrement the reference count for the dataset used by this experiment.
    """
    key = exp['data_key']
    global_dataset_cache[key]['refcount'] -= 1
    if global_dataset_cache[key]['refcount'] <= 0:
        del global_dataset_cache[key]
        
def sanitize_filename(name):
    """移除文件名中的非法字符"""
    return re.sub(r'[\/:*?"<>|]', '_', name)

def run_plots_optimized(plots, base_data):
    """
    Execute each plot sequentially.
    """
    usage = preload_experiment_data(plots)
    
    for plot_index, plot in enumerate(plots, 1):
        base_year = plot['base_year']
        experiments = plot['experiments']

        # 加载实验数据
        for exp in experiments:
            get_experiment_data(exp, usage)

        plot_exps = []
        for exp in experiments:
            plot_exps.append({
                'label': exp['label'],
                'U': exp['U_data'],
                'T': exp['T_data'],
                'x_offset': exp['x_offset'],
                'line_color': exp.get('line_color', None),
                'fill_color': exp.get('fill_color', None)
            })

        # 分配默认颜色
        plot_exps = assign_default_colors(plot_exps)

        # 拼接实验标签以形成唯一文件名
        experiment_labels = "_".join(sanitize_filename(exp['label']) for exp in experiments)
        save_path = f'/home/weiji/restart_exam/20250221/50hpawind/{base_year}_plot{plot_index}_{experiment_labels}'

        # 调用绘图函数
        plot_experiments_UT(
            base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
            base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
            plot_exps, base_year,
            save_path=save_path
        )

        # 释放实验数据
        for exp in experiments:
            release_experiment_data(exp)
##############################
# Pre-compute Base Data
##############################

base_file = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'
base_years = ["0008", "0003", "0013", "0014", "0019"]
base_data = {}
for year in base_years:
    base_data[year] = {}
    base_data[year]['ref_U'], base_data[year]['clim_U'] = process_U_base(
        base_file, lat=60, plev=5000, months=[1,2,3,4,5,6], base_year=year
    )
    base_data[year]['ref_T'], base_data[year]['clim_T'] = process_T_base(
        base_file, plev=5000, lat_range=(60,90), months=[1,2,3,4,5,6], base_year=year
    )

##############################
# Define Plots Configuration (All 20 Plots)
##############################

plots = [
    # Plot 1: Validation plot for 0008Feb NOCOUPL_reference and 0008Feb NOCOUPL (base year: 0008)
    {
        'base_year': '0008',
        'experiments': [
            {
                'label': '0008Feb_NOCOUPL_reference',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/*.nc',
                'x_offset': 31
            },
            {
                'label': '0008Feb_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc',
                'x_offset': 31
            }
        ]
    },
    # Plot 2: 0008Jan small, 0008Feb small, 0008Mar small (base year: 0008)
    {
        'base_year': '0008',
        'experiments': [
            {
                'label': '0008Jan_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/*.nc',
                'x_offset': 0
            },
            {
                'label': '0008Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0008Mar_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 3: 0008Feb large, 0008Mar large, 0008Apr large (base year: 0008)
    {
        'base_year': '0008',
        'experiments': [
            {
                'label': '0008Feb_large_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/*.nc',
                'x_offset': 31
            },
            {
                'label': '0008Mar_large_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/*.nc',
                'x_offset': 59
            },
            {
                'label': '0008Apr_large_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/*.nc',
                'x_offset': 89
            }
        ]
    },
    # Plot 4: 0008Feb small, 0008Feb large, 0008Feb water vapor (base year: 0008)
    {
        'base_year': '0008',
        'experiments': [
            {
                'label': '0008Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0008Feb_large_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/*.nc',
                'x_offset': 31
            },
            {
                'label': '0008Feb_vapor_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/*.nc',
                'x_offset': 31
            }
        ]
    },
    # Plot 5: 0003Feb small, 0003Mar small (base year: 0003)
    {
        'base_year': '0003',
        'experiments': [
            {
                'label': '0003Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0003/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0003/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0003Mar_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0003/Mar/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0003/Mar/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 6: 0013Feb small, 0013Mar small (base year: 0013)
    {
        'base_year': '0013',
        'experiments': [
            {
                'label': '0013Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0013Mar_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Mar/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Mar/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 7: 0014Feb small, 0014Mar small (base year: 0014)
    {
        'base_year': '0014',
        'experiments': [
            {
                'label': '0014Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0014Mar_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Mar/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Mar/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 8: 0019Feb small, 0019Mar small (base year: 0019)
    {
        'base_year': '0019',
        'experiments': [
            {
                'label': '0019Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0019Mar_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Mar/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Mar/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 9: 0008 NOCOUPL experiments (base year: 0008)
    {
        'base_year': '0008',
        'experiments': [
            {
                'label': '0008Feb_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc',
                'x_offset': 31
            },
            {
                'label': '0008Mar_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 10: 0013 NOCOUPL experiments (base year: 0013)
    {
        'base_year': '0013',
        'experiments': [
            {
                'label': '0013Feb_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc',
                'x_offset': 31
            },
            {
                'label': '0013Mar_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 11: 0014 NOCOUPL experiments (base year: 0014)
    {
        'base_year': '0014',
        'experiments': [
            {
                'label': '0014Feb_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc',
                'x_offset': 31
            },
            {
                'label': '0014Mar_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 12: 0019 NOCOUPL experiments (base year: 0019)
    {
        'base_year': '0019',
        'experiments': [
            {
                'label': '0019Feb_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc',
                'x_offset': 31
            },
            {
                'label': '0019Mar_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 13: 0008Feb small vs 0008Feb NOCOUPL (base year: 0008)
    {
        'base_year': '0008',
        'experiments': [
            {
                'label': '0008Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0008Feb_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc',
                'x_offset': 31
            }
        ]
    },
    # Plot 14: 0008Mar small vs 0008Mar NOCOUPL (base year: 0008)
    {
        'base_year': '0008',
        'experiments': [
            {
                'label': '0008Mar_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/*.nc',
                'x_offset': 59
            },
            {
                'label': '0008Mar_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 15: 0013Feb small vs 0013Feb NOCOUPL (base year: 0013)
    {
        'base_year': '0013',
        'experiments': [
            {
                'label': '0013Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0013Feb_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc',
                'x_offset': 31
            }
        ]
    },
    # Plot 16: 0013Mar small vs 0013Mar NOCOUPL (base year: 0013)
    {
        'base_year': '0013',
        'experiments': [
            {
                'label': '0013Mar_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Mar/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0013/Mar/*.nc',
                'x_offset': 59
            },
            {
                'label': '0013Mar_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 17: 0014Feb small vs 0014Feb NOCOUPL (base year: 0014)
    {
        'base_year': '0014',
        'experiments': [
            {
                'label': '0014Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0014Feb_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc',
                'x_offset': 31
            }
        ]
    },
    # Plot 18: 0014Mar small vs 0014Mar NOCOUPL (base year: 0014)
    {
        'base_year': '0014',
        'experiments': [
            {
                'label': '0014Mar_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Mar/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0014/Mar/*.nc',
                'x_offset': 59
            },
            {
                'label': '0014Mar_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc',
                'x_offset': 59
            }
        ]
    },
    # Plot 19: 0019Feb small vs 0019Feb NOCOUPL (base year: 0019)
    {
        'base_year': '0019',
        'experiments': [
            {
                'label': '0019Feb_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Feb/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Feb/*.nc',
                'x_offset': 31
            },
            {
                'label': '0019Feb_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc',
                'x_offset': 31
            }
        ]
    },
    # Plot 20: 0019Mar small vs 0019Mar NOCOUPL (base year: 0019)
    {
        'base_year': '0019',
        'experiments': [
            {
                'label': '0019Mar_small_pert',
                'U_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Mar/*.nc',
                'T_path': '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0019/Mar/*.nc',
                'x_offset': 59
            },
            {
                'label': '0019Mar_NOCOUPL',
                'U_path': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc',
                'T_path': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc',
                'x_offset': 59
            }
        ]
    }
]

##############################
# Run Optimized Plots
##############################

run_plots_optimized(plots, base_data)


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import math
import cftime

# ----------------------------------------------------
# Global paths and file references
# ----------------------------------------------------
MERGED_FILE = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'

# ----------------------------------------------------
# Data Processing Functions
# ----------------------------------------------------
def process_variable(ds, var):
    """
    对 O3、T、U 进行预处理：
    - O3：对经度求平均，再在 60–90°N 进行余弦加权平均；
    - T：对经度求平均，再在 60–90°N 选取并取最小值；
    - U：对经度求平均，然后选取 60°N 附近的值（nearest）。
    """
    if var == 'O3':
        da = ds['O3'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=slice(60, 90))
        weights = np.cos(np.deg2rad(da.lat))
        da = da.weighted(weights).mean(dim='lat')
    elif var == 'T':
        da = ds['T'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=slice(60, 90)).min(dim='lat')
    elif var == 'U':
        da = ds['U'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=60, method='nearest')
    else:
        raise ValueError("Unsupported variable: " + var)
    return da

def create_daily_data(da, start_month=1, end_month=5):
    """
    筛选 start_month 到 end_month 内的时间，返回每日数据。
    da = da.transpose('plev', 'time')会返回 (time_vals, plev_vals, data_array)，数据形状为 (plev, time)。
    """
    da = da.sel(time=da.time.dt.month.isin(range(start_month, end_month+1)))
    da = da.transpose('plev', 'time')
    time_vals = da['time'].values
    plev_vals = da['plev'].values
    data_array = da.values
    return time_vals, plev_vals, data_array

def read_climatology(var):
    """
    读取 MERGED_FILE 中所有年份的数据，计算 1-5 月每日平均值作为气候态。
    返回 dims=(plev, time) 的 DataArray。
    """
    ds = xr.open_dataset(MERGED_FILE)
    da = process_variable(ds, var)
    ds.close()
    # 筛选 1-5 月数据
    da = da.sel(time=da.time.dt.month.isin(range(1, 6)))
    calendar = ds.time.encoding.get('calendar', 'standard')
    # 计算所有年份的每日平均值
    da_clim = da.groupby('time.dayofyear').mean(dim='time')
    # 生成虚拟时间坐标（2000年 1-5 月）
    da_clim = da_clim.transpose('plev', 'dayofyear')
    #print(da.dims)  # Should be ('plev', 'time')
    #print(da.shape)  # Should be (23, <some_time_length>)
    days = da_clim['dayofyear'].values
    # 使用与实验数据相同的日历生成时间数组（参考年份设为2000）
    time_vals = [cftime.num2date(d - 1, units='days since 2000-01-01', calendar=calendar) for d in days]
    plev_vals = da_clim['plev'].values
    data_arr = da_clim.values
    clim_da = xr.DataArray(
        data_arr,
        coords={'plev': plev_vals, 'time': time_vals},
        dims=['plev', 'time']
    )
    return clim_da

def read_year_data(var, year):
    """
    读取 MERGED_FILE 中指定年份的 1-5 月数据。
    返回 dims=(plev, time) 的 DataArray。
    """
    ds = xr.open_dataset(MERGED_FILE)
    ds_year = ds.sel(time=ds.time.dt.year == int(year))
    da = process_variable(ds_year, var)
    ds.close()
    time_vals, plev_vals, data_arr = create_daily_data(da, 1, 5)
    year_da = xr.DataArray(
        data_arr,
        coords={'plev': plev_vals, 'time': time_vals},
        dims=['plev', 'time']
    )
    return year_da

def read_experiment_data(var, file_pattern, start_month=1):
    """
    打开实验数据文件，处理变量后返回从 start_month 到 5 月的每日数据。
    返回 dims=(plev, time) 的 DataArray。
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    da = process_variable(ds, var)
    ds.close()
    time_vals, plev_vals, data_arr = create_daily_data(da, start_month, 5)
    exp_da = xr.DataArray(
        data_arr,
        coords={'plev': plev_vals, 'time': time_vals},
        dims=['plev', 'time']
    )
    return exp_da


def get_anomaly(data_da, clim_da):
    # 将时间转换为 "%m-%d" 格式，忽略年份
    data_time_str = data_da.time.dt.strftime("%m-%d").values
    clim_time_str = clim_da.time.dt.strftime("%m-%d").values
    
    # 找到共同的月份和日子
    common_time_str = np.intersect1d(data_time_str, clim_time_str)
    
    # 选择 data_da 中对应共同月份和日子的数据
    data_da_common = data_da.sel(time=data_da.time.dt.strftime("%m-%d").isin(common_time_str))
    
    # 选择 clim_da 中对应共同月份和日子的数据
    clim_da_common = clim_da.sel(time=clim_da.time.dt.strftime("%m-%d").isin(common_time_str))
    
    # 计算异常值
    anom_vals = data_da_common.values - clim_da_common.values
    
    # 创建异常值的 DataArray
    anom_da = xr.DataArray(
        anom_vals,
        coords={'plev': data_da_common.plev, 'time': clim_da_common.time},
        dims=['plev', 'time']
    )
    return anom_da
# ----------------------------------------------------
# Plotting Functions
# ----------------------------------------------------
def auto_subplot_layout(n):
    """返回 (rows, cols) 用于 n 个子图，尽可能接近正方形排列。"""
    cols = int(math.ceil(math.sqrt(n)))
    rows = int(math.ceil(n / cols))
    return rows, cols

def plot_composite(runs, ref_run, clim_da, var, composite_name, year, save_path):
    """
    绘制所有 run 和 REF 的异常场，叠加气候态等值线。
    所有变量使用 5 条等值线，X 轴显示月份名称。
    """
    all_runs = runs + [ref_run]

    # 1) 确定共同时间
    common_time = None
    for run in all_runs:
        if common_time is None:
            common_time = run['time']
        else:
            common_time = np.intersect1d(common_time, run['time'])
    
    # 同步裁剪时间和数据
    for r in all_runs:
        mask = np.isin(r['time'], common_time)
        r['time'] = r['time'][mask]
        r['data'] = r['data'][:, mask]
    
    clim_mask = np.isin(clim_da.time.values, common_time)
    clim_run = {
        'time': clim_da.time.values[clim_mask].copy(),
        'plev': clim_da.plev.values.copy(),
        'data': clim_da.values[:, clim_mask].copy()
    }

    # 2) 统一异常值色阶
    all_anom = np.concatenate([r['data'].ravel() for r in all_runs])
    vmax = np.nanmax(np.abs(all_anom))
    vmin = -vmax

    # 3) 设置子图
    total_subplots = len(all_runs)
    rows, cols = auto_subplot_layout(total_subplots)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), squeeze=False)

    subplot_idx = 0
    cf = None
    for run in all_runs:
        r_idx = subplot_idx // cols
        c_idx = subplot_idx % cols
        ax = axes[r_idx][c_idx]
        if len(run['time']) > 0:
            time_mpl = mdates.date2num(run['time'])
            cf = ax.contourf(time_mpl, run['plev'], run['data'], levels=20,
                             cmap='RdBu_r', vmin=vmin, vmax=vmax)
            
            # 叠加气候态等值线（5 条）
            clim_data = clim_run['data']
            clim_min = np.nanmin(clim_data)
            clim_max = np.nanmax(clim_data)
            clim_levels = np.linspace(clim_min, clim_max, 5)
            CS = ax.contour(mdates.date2num(clim_run['time']), clim_run['plev'], clim_data,
                            levels=clim_levels, colors='k', linewidths=1.5)
            ax.clabel(CS, inline=True, fontsize=8, fmt='%.3e')
            
            ax.set_yscale('log')
            ax.invert_yaxis()
            ax.set_xlim(time_mpl[0], time_mpl[-1])
            ax.xaxis.set_major_locator(mdates.MonthLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))  # 显示 "Feb", "Mar" 等
        ax.set_ylabel('Pressure (Pa)', fontsize=8)
        ax.set_title(f"{run['label']} ({run['year']})", fontsize=9)
        subplot_idx += 1

    # 隐藏多余子图
    for i in range(subplot_idx, rows*cols):
        r_idx = i // cols
        c_idx = i % cols
        axes[r_idx][c_idx].axis('off')
    
    if cf is not None:
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(cf, cax=cax, label=f'{var} Anomaly')
    
    fig.suptitle(f"{composite_name} {var} Anomaly, Year {year}", fontsize=12)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"[INFO] Saved composite figure: {save_path}")

# ----------------------------------------------------
# 定义 composite groups（20组）及辅助构造文件路径
# ----------------------------------------------------
def get_year_small_pert(year):
    base = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    file_Feb = os.path.join(base, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    file_Mar = os.path.join(base, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    return file_Feb, file_Mar

def get_year_nocouple(year):
    file_Feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    file_Mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    return file_Feb, file_Mar

file_0008_Jan_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'
file_0008_Feb_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Feb_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Apr_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/BWCN.e122.f19_g16.002.0008-04.*.nc'
file_0008_Feb_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc'

composite_groups = [
    {'name': 'Composite 1', 'year': '0008', 'runs': [
        {'label': 'Jan small pertlim', 'file': file_0008_Jan_small, 'start_month': 1},
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small, 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small, 'start_month': 3}
    ]},
    {'name': 'Composite 2', 'year': '0008', 'runs': [
        {'label': 'Feb large pertlim', 'file': file_0008_Feb_large, 'start_month': 2},
        {'label': 'Mar large pertlim', 'file': file_0008_Mar_large, 'start_month': 3},
        {'label': 'Apr large pertlim', 'file': file_0008_Apr_large, 'start_month': 4}
    ]},
    {'name': 'Composite 3', 'year': '0008', 'runs': [
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small, 'start_month': 2},
        {'label': 'Feb large pertlim', 'file': file_0008_Feb_large, 'start_month': 2},
        {'label': 'Feb moist pertlim', 'file': file_0008_Feb_moist, 'start_month': 2}
    ]},
    {'name': 'Composite 4', 'year': '0008', 'runs': [
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small, 'start_month': 3},
        {'label': 'Mar large pertlim', 'file': file_0008_Mar_large, 'start_month': 3},
        {'label': 'Mar moist pertlim', 'file': file_0008_Mar_moist, 'start_month': 3}
    ]},
    {'name': 'Composite 5', 'year': '0003', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0003')[0], 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0003')[1], 'start_month': 3}
    ]},
    {'name': 'Composite 6', 'year': '0013', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0013')[0], 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0013')[1], 'start_month': 3}
    ]},
    {'name': 'Composite 7', 'year': '0014', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0014')[0], 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0014')[1], 'start_month': 3}
    ]},
    {'name': 'Composite 8', 'year': '0019', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0019')[0], 'start_month': 2},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0019')[1], 'start_month': 3}
    ]},
    {'name': 'Composite 9', 'year': '0008', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc', 'start_month': 2},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 10', 'year': '0013', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc', 'start_month': 2},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 11', 'year': '0014', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc', 'start_month': 2},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 12', 'year': '0019', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc', 'start_month': 2},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 13', 'year': '0008', 'runs': [
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small, 'start_month': 2},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc', 'start_month': 2}
    ]},
    {'name': 'Composite 14', 'year': '0008', 'runs': [
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small, 'start_month': 3},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 15', 'year': '0013', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0013')[0], 'start_month': 2},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc', 'start_month': 2}
    ]},
    {'name': 'Composite 16', 'year': '0013', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0013')[1], 'start_month': 3},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 17', 'year': '0014', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0014')[0], 'start_month': 2},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc', 'start_month': 2}
    ]},
    {'name': 'Composite 18', 'year': '0014', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0014')[1], 'start_month': 3},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc', 'start_month': 3}
    ]},
    {'name': 'Composite 19', 'year': '0019', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0019')[0], 'start_month': 2},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc', 'start_month': 2}
    ]},
    {'name': 'Composite 20', 'year': '0019', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0019')[1], 'start_month': 3},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc', 'start_month': 3}
    ]}
]

def generate_all_composites():
    output_dir = '/home/weiji/restart_exam/plots/Vertical_Profiles_new/'
    variables = ['O3', 'T', 'U']
    for var in variables:
        print(f"\n[INFO] Generating composite plots for {var}...")
        clim_da = read_climatology(var)
        for group in composite_groups:
            year = group['year']
            composite_name = group['name']
            runs_info = group['runs']
            # 读取 REF 数据并计算异常值
            ref_da = read_year_data(var, year)
            ref_anom = get_anomaly(ref_da, clim_da)
            ref_run = {
                'label': 'REF',
                'year': year,
                'time': ref_anom.time.values,
                'plev': ref_anom.plev.values,
                'data': ref_anom.values
            }
            # 读取实验数据并计算异常值
            exp_runs = []
            for r in runs_info:
                exp_da = read_experiment_data(var, r['file'], r['start_month'])
                exp_anom = get_anomaly(exp_da, clim_da)
                exp_runs.append({
                    'label': r['label'],
                    'year': year,
                    'time': exp_anom.time.values,
                    'plev': exp_anom.plev.values,
                    'data': exp_anom.values
                })
            save_path = os.path.join(output_dir, f"{composite_name.replace(' ', '_')}_{var}.png")
            plot_composite(exp_runs, ref_run, clim_da, var, composite_name, year, save_path)
    print("\n[INFO] All composite plots have been generated.")

# 运行生成复合图的函数
generate_all_composites()

def plot_climatology(clim_da, var, save_path):
    """
    绘制气候态臭氧的垂直剖面填色图，与异常图设置一致。
    """
    fig, ax = plt.subplots(figsize=(6, 4))
    time_mpl = mdates.date2num(clim_da.time.values)  # 将时间转换为 matplotlib 可识别格式
    cf = ax.contourf(time_mpl, clim_da.plev, clim_da.values, levels=20, cmap='viridis')  # 绘制填色图
    ax.set_yscale('log')  # Y 轴为对数刻度
    ax.invert_yaxis()  # 反转 Y 轴（压力从大到小）
    ax.set_xlim(time_mpl[0], time_mpl[-1])  # 设置 X 轴范围
    ax.xaxis.set_major_locator(mdates.MonthLocator())  # X 轴显示每月
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))  # X 轴显示 "Jan", "Feb" 等
    ax.set_ylabel('Pressure (Pa)', fontsize=10)  # Y 轴标签
    ax.set_title(f'Climatology of {var}', fontsize=12)  # 标题
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # 添加色条位置
    fig.colorbar(cf, cax=cax, label=f'{var} Climatology')  # 添加色条
    plt.savefig(save_path, dpi=150, bbox_inches='tight')  # 保存图形
    plt.close(fig)  # 关闭图形，避免内存占用
    print(f"[INFO] Saved climatology figure: {save_path}")

# 调用函数（假设在主程序或 generate_all_composites 中）
clim_da = read_climatology('O3')  # 获取气候态臭氧数据
output_dir = '/home/weiji/restart_exam/plots/Vertical_Profiles_new/'
save_path_clim = os.path.join(output_dir, "Climatology_O3.png")  # 设置保存路径
plot_climatology(clim_da, 'O3', save_path_clim)  # 绘制并保存